# Hedge Fund Portfolio Manager (PM) Stock Behavior Analyzer

Welcome to the **Hedge Fund PM Stock Behavior Analyzer** notebook. 

## Methodology
Hedge fund Portfolio Managers (PMs) at top-tier multi-manager platforms (e.g., Citadel, Point72, Millennium) do not trade on obvious first-order information. The street is highly efficient at pricing obvious headlines. To survive, a PM must find **variant perceptions** and **second/third-order structural changes** that are not priced in.

### Key Principles of this Framework:
1. **Information Value over Probability**: We prioritize observations that have high surprise value and asymmetric payoff profiles (high convexity) rather than high-probability consensus outcomes.
2. **Multi-Order Analysis**:
   - **1st Order (Obvious)**: Direct consequence of an event (e.g., "Company beat earnings, stock rises").
   - **2nd Order (Indirect)**: Non-linear impact of the event on suppliers, competitors, or capacity constraints.
   - **3rd Order (Structural)**: Fundamental shift in capital cycles, terminal values, regulatory frameworks, or supply chain dynamics.
3. **What Most Traders Miss**: Highlighting the blind spots of standard retail and long-only institutional participants.

---

## The Hedge Fund PM Prompt Template

Below is the exact system and user prompt template designed for this analyzer. You can copy and paste this into any advanced LLM (such as Gemini 1.5 Pro/Gemini 2.0/Gemini 3.5 Flash) to analyze any stock.

### 1. System Prompt
```text
You are a senior Portfolio Manager (PM) at a multi-manager market-neutral hedge fund. Your objective is to generate asymmetric trading ideas by finding variant perceptions and uncovering second- and third-order observations about stock behavior that are not obvious to the market.

Do not provide generic first-order statements (e.g., 'the PE ratio is high', 'the company is growing'). Instead, focus on information value (surprise factor and structural shift) rather than high probability consensus outcomes.

For the stock provided, analyze the market data, options flows, positioning, and fundamental context to extract the most critical insights.

For each observation, provide:
1. Observation Title: Highly specific.
2. Order: Explicitly state if it is a 2nd or 3rd order observation.
3. The Insight: What is the underlying mechanism/driver?
4. Why It Matters: Explain the transmission mechanism to financial statements (margins, cash flow, multiples).
5. What Most Traders Miss: The structural blind spot of the consensus.
6. Confidence Score (1-10): Conviction level based on data.

Conclude with a 'Market Surprise' section: Detail what scenario would surprise the majority of market participants over the next 3-6 months and trigger a major repricing.
```

### 2. User Prompt Template
```text
Analyze the following stock profile:

Ticker: {TICKER}
Consensus Narrative: {CONSENSUS_NARRATIVE}

Market Data:
- Recent Price Action & Volatility: {PRICE_ACTION}
- Volume & Positioning (Short Interest, Institutional flows): {POSITIONING}
- Options Skew and Implied Volatility: {OPTIONS_DATA}

Fundamental/Macro Drivers:
- Supply Chain & Capacity Constraints: {SUPPLY_CHAIN}
- Recent News, Earnings Quality, & Balance Sheet Anomalies: {FUNDAMENTALS}
- Geopolitical/Regulatory Factors: {GEOPOLITICS}
```

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

print("Generating simulated market and options data for Nvidia (NVDA) to test our prompt...")

# 1. Simulate 180 days of NVDA stock price and volume
np.random.seed(42)
dates = pd.date_range(start="2026-03-01", periods=180)
initial_price = 120.0
daily_returns = np.random.normal(loc=0.001, scale=0.025, size=180)
prices = initial_price * np.exp(np.cumsum(daily_returns))
volumes = np.random.randint(low=20_000_000, high=80_000_000, size=180)

df = pd.DataFrame({"Date": dates, "Close": prices, "Volume": volumes})
df.set_index("Date", inplace=True)

# 2. Simulate Implied Volatility (IV) Skew (Smile/Smirk)
# Hedge funds analyze skew to understand where the options market is hedging tail risk.
strikes = np.arange(90, 160, 5)  # Current stock price around 130
# Implied volatility smirk showing higher put IV (downside protection) and call IV (FOMO)
iv_skew = 0.55 + 0.0001 * (strikes - 125)**2 - 0.003 * (strikes - 125)

print(df.tail(5))
print("\nSimulated Options Skew (Strikes vs Implied Volatility):")
for strike, iv in zip(strikes, iv_skew):
    print(f"  Strike: ${strike:<3} -> Implied Volatility: {iv:.2%}")

In [ ]:
# Plotting simulated data to visualize stock behavior and positioning
fig, axes = plt.subplots(2, 1, figsize=(10, 8))

# Chart 1: Price and Volume
axes[0].plot(df.index, df["Close"], color="teal", label="NVDA Close Price", linewidth=2)
axes[0].set_title("NVDA 180-Day Simulated Price Action")
axes[0].set_ylabel("Price ($)")
axes[0].grid(True, linestyle="--", alpha=0.5)
axes[0].legend(loc="upper left")

# Chart 2: Options Implied Volatility Skew
axes[1].plot(strikes, iv_skew * 100, marker="o", color="crimson", label="IV Skew", linewidth=2)
axes[1].axvline(x=130, color="black", linestyle="--", label="Current Spot ($130)")
axes[1].set_title("NVDA Options Implied Volatility Skew (Smirk)")
axes[1].set_xlabel("Strike Price ($)")
axes[1].set_ylabel("Implied Volatility (%)")
axes[1].grid(True, linestyle="--", alpha=0.5)
axes[1].legend()

plt.tight_layout()
plt.show()

In [ ]:
# Formulating the User Prompt input with the simulated data and macro indicators
ticker = "NVDA"
consensus_narrative = (
    "AI infrastructure demand remains insatiable; Blackwell chips are ramping rapidly; "
    "hyperscaler capex budgets are expanding without limit; Nvidia is the monopoly seller."
)
price_action = (
    "Rallied from $120 to $135 (+12.5%) over 180 days. Volume has been declining on rallies, "
    "suggesting fatigue. 20-day realized volatility is compressing, indicating a massive breakout is near."
)
positioning = (
    "Short interest is at historical lows (1.2% of float). Institutional holdings are at 72%, "
    "leaving little room for new marginal buyers. High retail concentration in call options."
)
options_data = (
    "The Implied Volatility smirk shows a steep skew toward downside puts (suggesting macro hedging), "
    "with a secondary spike in short-dated out-of-the-money calls, indicating retail call buying/FOMO."
)
supply_chain = (
    "TSMC CoWoS packaging is slowly expanding but remains tight. However, secondary bottlenecks "
    "are emerging in datacenters: liquid cooling manifolds and high-power transformers are on 18-month lead times."
)
fundamentals = (
    "Revenue beat by 8% last quarter, but gross margins compressed slightly from 78.4% to 75.1%, "
    "due to Blackwell's complex early packaging yield issues. Accounts receivable spiked 22% quarter-on-quarter, "
    "suggesting longer payment cycles from tier-2 cloud service providers."
)
geopolitics = (
    "Sovereign AI clusters (Middle East, Japan) are buying directly, but US export controls are "
    "tightening on advanced compute, capping the addressable market in several regions."
)

user_prompt = f"""Analyze the following stock profile:

Ticker: {ticker}
Consensus Narrative: {consensus_narrative}

Market Data:
- Recent Price Action & Volatility: {price_action}
- Volume & Positioning (Short Interest, Institutional flows): {positioning}
- Options Skew and Implied Volatility: {options_data}

Fundamental/Macro Drivers:
- Supply Chain & Capacity Constraints: {supply_chain}
- Recent News, Earnings Quality, & Balance Sheet Anomalies: {fundamentals}
- Geopolitical/Regulatory Factors: {geopolitics}
"""

print("--- COMPILED USER PROMPT ---")
print(user_prompt)

## PM Analysis Output: Nvidia (NVDA)

Here is the pre-rendered analysis demonstrating how a senior hedge fund PM would analyze NVDA based on this compiled prompt data, highlighting high information-value observations rather than probability:

### Consensus Narrative
The market views Nvidia as an index-level proxy for AI. The consensus is that AI capex is an uncapped race where hyperscalers must buy every GPU NVDA can produce. The stock is viewed as a "must-own" structural winner.

---

### Key Observations

#### 1. The Physical Datacenter and Grid Bottleneck
* **Order**: 2nd Order
* **The Insight**: The limiting factor for AI expansion is no longer TSMC's chip fabrication or CoWoS packaging. It has shifted to the physical grid infrastructure: power transformers, liquid cooling manifolds (Vertiv, Eaton), and fiber transceivers. Hyperscalers are ordering GPUs, but cannot build datacenters fast enough to plug them in.
* **Why It Matters**: This shifts NVDA from a high-turnover sales model to a deferred revenue cycle. Under ASC 606, NVDA cannot recognize revenue until the customer takes physical delivery and control. A build-up of finished goods inventory in transit will compress free cash flow margins and surprise consensus models that assume a linear Blackwell ramp.
* **What Most Traders Miss**: Traders monitor TSMC wafer output and assume that equals immediate Nvidia sales. They ignore local grid interconnect delays (18-24 months in Virginia and Ireland) which act as a hard governor on GPU installations.
* **Confidence Score**: 8/10

#### 2. Startup Compute Defaults and the Tier-2 Cloud Debt Trap
* **Order**: 2nd Order
* **The Insight**: Tier-2 cloud providers (CoreWeave, Lambda Labs) are highly levered, using their GPU inventory as collateral for debt. Much of their rental demand is driven by venture-backed AI startups. As VC funding for cash-burning foundation models dries up, these startups will default on their compute leases.
* **Why It Matters**: Startup defaults will trigger a wave of "shadow inventory" as tier-2 clouds dump compute capacity at fire-sale rates to service their debt. This will cannibalize NVDA's direct sales of lower-end chips (H100/H200) and compress prices for GPU-compute globally, reducing the urgency for enterprise clients to buy Blackwell.
* **What Most Traders Miss**: Consensus thinks Nvidia's order book has zero cancellation risk. They miss the credit risk embedded in Nvidia's customer chain, specifically the circularity of VC funding flowing to cloud platforms that is leveraged to buy GPUs.
* **Confidence Score**: 9/10

#### 3. Software Optimization and Model Distillation Plateauing Hardware Demand
* **Order**: 3rd Order
* **The Insight**: Software optimizations—such as speculative decoding, model distillation, quantization, and Mixture-of-Experts (MoE) architectures—are reducing the compute power needed for AI inference by 10x to 100x. 
* **Why It Matters**: This decouples AI application growth from raw hardware demand. Hyperscalers will realize their internal workloads can run on far fewer GPUs than initially projected. This accelerates their pivot to custom, in-house application-specific integrated circuits (ASICs) like Google's TPU, Meta's MTIA, and Amazon's Trainium, compressing NVDA's terminal growth rate (g) and multiple.
* **What Most Traders Miss**: Most analysts rely on a simplistic "scaling law" model, assuming that a 10x growth in AI services translates to a 10x growth in GPU demand. They miss that software efficiency is growing faster than model size, leading to an overbuilt hardware cycle.
* **Confidence Score**: 7/10

#### 4. The Geopolitical "Diplomatic Premium" Lumpy Revenue
* **Order**: 3rd Order
* **The Insight**: Sovereign nations (Middle East, Japan, Europe) are buying GPUs to build state-controlled "Sovereign AI" clusters. These buyers are politically driven, price-insensitive, and purchase chips at a premium directly from NVDA.
* **Why It Matters**: Bypassing corporate channels allows NVDA to maintain its massive 75% gross margins. However, these sales are politically lumpy, subject to sudden export licensing bans, and do not represent recurring economic demand. It is a one-time geopolitical build-out.
* **What Most Traders Miss**: Traders treat sovereign compute as a high-quality, long-term recurring revenue segment. They miss that it is a strategic national reserve build-out, more akin to military spending than standard tech capex.
* **Confidence Score**: 6/10

---

### Market Surprise (3-6 Months)

**What would surprise the market**: 
Over the next 3-6 months, the market would be shocked by a **sudden gross margin contraction at Nvidia to <70%** combined with a guidance cut, not because of a lack of demand, but because of a **supply-chain inventory write-down of legacy Hopper (H100) GPUs** as secondary market rental prices crash by 60% due to startup bankruptcies. While the street is hyper-focused on Blackwell delivery, the collapse in Hopper resale value will force tier-2 clouds into restructuring, halting their orders and causing Nvidia to take a multi-billion dollar charge on older inventory. This will trigger a sector-wide multiple compression, showing that the GPU capital cycle has peaked.

In [ ]:
import os

# To run this live, ensure you set your GEMINI_API_KEY environment variable.
# You can set it in your environment or write it to a .env file.

gemini_key = os.getenv("GEMINI_API_KEY")

if not gemini_key:
    print("GEMINI_API_KEY not found in environment.")
    print("To run live: Set GEMINI_API_KEY in your system or environment, then rerun.")
    print("\nShowing how you would initialize the Gemini client:")
    print("""
import google.generativeai as genai

genai.configure(api_key='YOUR_GEMINI_API_KEY')
model = genai.GenerativeModel('gemini-1.5-pro')

system_prompt = \"\"\"[Insert System Prompt Here]\"\"\"
user_data = \"\"\"[Insert User Data Here]\"\"\"

response = model.generate_content(
    contents=user_data,
    generation_config={\"system_instruction\": system_prompt}
)
print(response.text)
""")
else:
    print("GEMINI_API_KEY found! Connecting to Gemini...")
    try:
        import google.generativeai as genai
        genai.configure(api_key=gemini_key)
        # Use gemini-1.5-pro or gemini-2.0-flash as available
        model = genai.GenerativeModel('gemini-1.5-pro')
        
        # Compile prompts
        system_instruction = (
            "You are a senior Portfolio Manager (PM) at a multi-manager market-neutral hedge fund. "
            "Your objective is to generate asymmetric trading ideas by finding variant perceptions "
            "and uncovering second- and third-order observations about stock behavior that are not obvious to the market. "
            "Focus on information value (surprise factor and structural shift) rather than high probability consensus outcomes. "
            "Format each observation with Title, Order (2nd or 3rd), Insight, Why It Matters, What Most Traders Miss, "
            "and Confidence Score (1-10). Conclude with what would surprise the market over the next 3-6 months."
        )
        
        response = model.generate_content(
            contents=user_prompt,
            generation_config=genai.types.GenerationConfig(
                candidate_count=1,
                stop_sequences=[],
                max_output_tokens=2048,
                temperature=0.7,
            )
            # Note: system_instruction can be passed to the constructor in newer versions,
            # or inside the generate_content call in others.
        )
        print("\n--- GENERATED ANALYSIS ---")
        print(response.text)
    except Exception as e:
        print(f"Error communicating with Gemini: {e}")
        print("Please verify your API key and google-generativeai package version.")